In [19]:
import duckdb
import pandas as pd

##### View Raw Data

In [28]:
## ---- check tables ----
# Connect to your local database
with duckdb.connect(database="duckdb.db", read_only=True) as con:
    # SHOW ALL TABLES returns the database, schema, name, and column names
    catalog_df = con.execute("SHOW ALL TABLES").fetchdf()

# Display the results
catalog_df[['name', 'column_names']]

,name,column_names
0,agg_daily_cc_riders,"[pickup_day, total_cc_riders]"
1,fct_taxi_financials,"[fare_amount, tip_amount, tolls_amount, total_..."
2,stg_taxi_rides,"[vendor_id, pickup_datetime, dropoff_datetime,..."
3,taxi_rides_raw,"[VendorID, tpep_pickup_datetime, tpep_dropoff_..."
4,total_amounts,"[fare_amount, tip_amount, tolls_amount, total_..."
5,total_creditcard_riders_by_day,"[day, total_cc_riders]"


In [30]:
# --- view the schema of a table ---
with duckdb.connect(database="duckdb.db", read_only=True) as con:
    # Let's inspect the Gold layer table you just built
    schema_df = con.execute("DESCRIBE agg_daily_cc_riders").fetchdf()
schema_df

,column_name,column_type,null,key,default,extra
0,pickup_day,TIMESTAMP,YES,None,None,None
1,total_cc_riders,BIGINT,YES,None,None,None


In [20]:
df = pd.DataFrame()

with duckdb.connect(database="duckdb.db", read_only=True) as con:
    df = con.execute("SELECT * FROM taxi_rides_raw LIMIT 100").fetchdf()

df.sample(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
61,1,2026-01-01 00:59:22,2026-01-01 01:07:58,0,1.30,1,N,75,151,2,10.0,1.00,0.5,0.00,0.0,1.0,12.50,0.0,0.00,0.00
6,1,2026-01-01 00:17:54,2026-01-01 00:28:32,1,1.30,1,N,142,50,2,11.4,4.25,0.5,0.00,0.0,1.0,17.15,2.5,0.00,0.75
66,2,2026-01-01 00:29:56,2026-01-01 00:53:41,1,15.42,1,N,132,255,1,56.9,1.00,0.5,14.85,0.0,1.0,76.00,0.0,1.75,0.00
10,2,2026-01-01 00:12:46,2026-01-01 00:48:58,1,3.09,1,N,161,144,1,29.6,1.00,0.5,7.07,0.0,1.0,42.42,2.5,0.00,0.75
89,2,2026-01-01 00:17:46,2026-01-01 00:30:00,1,2.98,1,N,170,236,1,14.2,1.00,0.5,4.99,0.0,1.0,24.94,2.5,0.00,0.75


In [22]:
with duckdb.connect(database="duckdb.db", read_only=True) as con:
    # This query finds the values that are NOT in your allowed list
    bad_data = con.execute("""
        SELECT Passenger_count, count(*) 
        FROM taxi_rides_raw 
        WHERE Passenger_count NOT IN (1, 2, 3, 4, 5, 6)
        GROUP BY 1
    """).fetchdf()
bad_data

,passenger_count,count_star()
0,0,14787
1,7,2
2,8,4
3,9,1


##### View staged data

In [25]:
with duckdb.connect(database="duckdb.db", read_only=True) as con:
    df = con.execute("SELECT * FROM stg_taxi_rides LIMIT 100").fetchdf()

df.sample(5)

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,fare_amount,tip_amount,tolls_amount,total_amount
57,2,2026-01-01 00:18:27,2026-01-01 00:29:21,3,1.49,11.4,4.29,0.0,21.44
58,2,2026-01-01 00:53:10,2026-01-01 01:14:30,2,2.57,19.1,4.97,0.0,29.82
67,2,2026-01-01 00:24:59,2026-01-01 00:55:26,1,4.75,29.6,0.00,0.0,35.35
46,2,2026-01-01 00:28:29,2026-01-01 00:46:04,3,4.78,23.3,2.00,0.0,27.80
62,2,2026-01-01 00:07:54,2026-01-01 00:19:37,1,5.02,21.9,0.00,0.0,31.15


#### Drop Orphaned Table
- even if we delete some old data models, dbt won't delete these in the data warehouse

In [ ]:

with duckdb.connect(database="duckdb.db") as con:
    # Drop the dummy models
    con.execute("DROP TABLE IF EXISTS my_first_dbt_model")
    con.execute("DROP VIEW IF EXISTS my_second_dbt_model")
    con.execute("DROP TABLE IF EXISTS total_creditcard_riders_by_day")
    con.execute("DROP VIEW IF EXISTS total_amounts")
    print("Orphaned models deleted!")

Orphaned models deleted!
